In [1]:
%pip install -q pyspark==4.0.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from google.colab import files

uploaded = files.upload()

Saving Sales_Data_Engineering_250000.zip to Sales_Data_Engineering_250000.zip


In [3]:
import zipfile
from pathlib import Path

zip_name = next(
    name for name in uploaded if name.endswith(".zip")
)

with zipfile.ZipFile(zip_name, "r") as archive:
    archive.extractall("/content/sales_challenge")

ROOT = Path("/content/sales_challenge/sales-project")

print("Sales files found:")
for file in sorted((ROOT / "data" / "raw").glob("orders_part_*.csv")):
    print(file.name)

Sales files found:
orders_part_01.csv
orders_part_02.csv
orders_part_03.csv
orders_part_04.csv
orders_part_05.csv


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("SalesDataEngineering")
    .config("spark.sql.shuffle.partitions", 8)
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.1


In [5]:
raw_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(ROOT / "data" / "raw" / "orders_part_*.csv"))
)

print("Total sales rows:", raw_orders.count())

raw_orders.printSchema()
raw_orders.show(5, truncate=False)

Total sales rows: 250000
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price_cents: string (nullable = true)
 |-- status: string (nullable = true)
 |-- currency: string (nullable = true)

+--------+-----------+----------+----------+--------+----------------+---------+--------+
|order_id|customer_id|product_id|order_date|quantity|unit_price_cents|status   |currency|
+--------+-----------+----------+----------+--------+----------------+---------+--------+
|O0179144|C07098     |P0126     |2025-06-11|4       |9648            |completed|CAD     |
|O0096822|C09576     |P0239     |2025-04-24|5       |2255            |completed|CAD     |
|O0055925|C02282     |P0283     |2025-12-19|4       |33910           |completed|CAD     |
|O0055549|C08505     |P0210     |2025-11-15|2       |6102            |cancelled|CAD     |
|O0184

In [6]:
raw_count = raw_orders.count()

deduped_orders = raw_orders.dropDuplicates()
deduped_count = deduped_orders.count()

duplicates_removed = raw_count - deduped_count

print("Raw rows:", raw_count)
print("Duplicates removed:", duplicates_removed)
print("Remaining rows:", deduped_count)

Raw rows: 250000
Duplicates removed: 1000
Remaining rows: 249000


In [7]:
from pyspark.sql import functions as F

repeated_ids = (
    deduped_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Repeated order IDs:", repeated_ids.count())

assert deduped_count == 249_000
assert repeated_ids.count() == 0

Repeated order IDs: 0


In [8]:
missing_counts = deduped_orders.select([
    F.sum(
        F.when(
            F.col(column).isNull() |
            (F.trim(F.col(column)) == ""),
            1
        ).otherwise(0)
    ).alias(column)
    for column in deduped_orders.columns
])

missing_counts.show(truncate=False)

+--------+-----------+----------+----------+--------+----------------+------+--------+
|order_id|customer_id|product_id|order_date|quantity|unit_price_cents|status|currency|
+--------+-----------+----------+----------+--------+----------------+------+--------+
|0       |249        |0         |0         |0       |0               |249   |0       |
+--------+-----------+----------+----------+--------+----------------+------+--------+



In [9]:
typed_orders = (
    deduped_orders
    .withColumn(
        "quantity_num",
        F.expr("try_cast(quantity as int)")
    )
    .withColumn(
        "price_cents_num",
        F.expr("try_cast(unit_price_cents as bigint)")
    )
    .withColumn(
        "order_date_parsed",
        F.expr("try_cast(order_date as date)")
    )
)

typed_orders.select(
    "quantity", "quantity_num",
    "unit_price_cents", "price_cents_num",
    "order_date", "order_date_parsed"
).show(5, truncate=False)

+--------+------------+----------------+---------------+----------+-----------------+
|quantity|quantity_num|unit_price_cents|price_cents_num|order_date|order_date_parsed|
+--------+------------+----------------+---------------+----------+-----------------+
|3       |3           |15436           |15436          |2025-05-07|2025-05-07       |
|1       |1           |43474           |43474          |2025-03-29|2025-03-29       |
|1       |1           |24528           |24528          |2025-01-10|2025-01-10       |
|2       |2           |33690           |33690          |2025-07-12|2025-07-12       |
|2       |2           |20102           |20102          |2025-03-14|2025-03-14       |
+--------+------------+----------------+---------------+----------+-----------------+
only showing top 5 rows


In [10]:
typed_orders.select(
    F.sum(F.when(
        F.col("quantity_num").isNull() |
        (F.col("quantity_num") <= 0), 1
    ).otherwise(0)).alias("invalid_quantity"),

    F.sum(F.when(
        F.col("price_cents_num").isNull() |
        (F.col("price_cents_num") < 0), 1
    ).otherwise(0)).alias("invalid_price"),

    F.sum(F.when(
        F.col("order_date_parsed").isNull() |
        (F.year("order_date_parsed") != 2025), 1
    ).otherwise(0)).alias("invalid_date")
).show()

+----------------+-------------+------------+
|invalid_quantity|invalid_price|invalid_date|
+----------------+-------------+------------+
|             249|          249|         249|
+----------------+-------------+------------+



In [11]:
customers = (
    spark.read.option("header", True)
    .csv(str(ROOT / "data/raw/customers.csv"))
)

products = (
    spark.read.option("header", True)
    .csv(str(ROOT / "data/raw/products.csv"))
)

assert customers.count() == 10_000
assert customers.select("customer_id").distinct().count() == 10_000

assert products.count() == 500
assert products.select("product_id").distinct().count() == 500

print("Customer and product keys are unique.")

Customer and product keys are unique.


In [12]:
customer_keys = customers.select("customer_id").withColumn(
    "customer_exists", F.lit(True)
)

product_keys = products.select("product_id").withColumn(
    "product_exists", F.lit(True)
)

checked_orders = (
    typed_orders
    .join(customer_keys, "customer_id", "left")
    .join(product_keys, "product_id", "left")
)

assert checked_orders.count() == deduped_count

In [13]:
validated_orders = checked_orders.withColumn(
    "rejection_reason",
    F.when(
        F.col("customer_exists").isNull(), "invalid_customer"
    )
    .when(
        F.col("product_exists").isNull(), "invalid_product"
    )
    .when(
        F.col("quantity_num").isNull() |
        (F.col("quantity_num") <= 0), "invalid_quantity"
    )
    .when(
        F.col("price_cents_num").isNull() |
        (F.col("price_cents_num") < 0), "invalid_price"
    )
    .when(
        F.col("order_date_parsed").isNull() |
        (F.year("order_date_parsed") != 2025) |
        (
            F.date_format("order_date_parsed", "yyyy-MM-dd")
            != F.col("order_date")
        ),
        "invalid_date"
    )
    .when(
        F.col("status").isNull() |
        ~F.col("status").isin("completed", "cancelled", "refunded"),
        "invalid_status"
    )
    .when(
        F.col("currency").isNull() |
        (F.col("currency") != "CAD"),
        "invalid_currency"
    )
)

In [14]:
clean_orders = validated_orders.filter(
    F.col("rejection_reason").isNull()
).drop("customer_exists", "product_exists", "rejection_reason")

rejected_orders = validated_orders.filter(
    F.col("rejection_reason").isNotNull()
).drop("customer_exists", "product_exists")

clean_count = clean_orders.count()
rejected_count = rejected_orders.count()

print("Valid orders:", clean_count)
print("Rejected orders:", rejected_count)

rejected_orders.groupBy("rejection_reason").count().show(
    truncate=False
)

assert clean_count == 247_506
assert rejected_count == 1_494
assert clean_count + rejected_count == deduped_count

Valid orders: 247506
Rejected orders: 1494
+----------------+-----+
|rejection_reason|count|
+----------------+-----+
|invalid_customer|249  |
|invalid_status  |249  |
|invalid_product |249  |
|invalid_price   |249  |
|invalid_date    |249  |
|invalid_quantity|249  |
+----------------+-----+



In [15]:
enriched_orders = (
    clean_orders
    .join(customers, "customer_id", "inner")
    .join(products, "product_id", "inner")
    .withColumn(
        "month",
        F.date_format("order_date_parsed", "yyyy-MM")
    )
    .withColumn(
        "revenue_cents",
        F.col("quantity_num").cast("long") *
        F.col("price_cents_num")
    )
)

In [16]:
enriched_count = enriched_orders.count()

print("Orders after joining:", enriched_count)

assert enriched_count == clean_count
assert (
    enriched_orders.select("order_id").distinct().count()
    == enriched_count
)

enriched_orders.select(
    "order_id",
    "region",
    "product_name",
    "month",
    "quantity_num",
    "price_cents_num",
    "revenue_cents",
    "status"
).show(5, truncate=False)

Orders after joining: 247506
+--------+------+------------+-------+------------+---------------+-------------+---------+
|order_id|region|product_name|month  |quantity_num|price_cents_num|revenue_cents|status   |
+--------+------+------------+-------+------------+---------------+-------------+---------+
|O0131751|West  |Product 222 |2025-05|3           |15436          |46308        |completed|
|O0021903|North |Product 175 |2025-03|1           |43474          |43474        |completed|
|O0158616|West  |Product 1   |2025-01|1           |24528          |24528        |refunded |
|O0050930|East  |Product 49  |2025-07|2           |33690          |67380        |refunded |
|O0103889|West  |Product 96  |2025-03|2           |20102          |40204        |completed|
+--------+------+------------+-------+------------+---------------+-------------+---------+
only showing top 5 rows


In [17]:
completed_orders = enriched_orders.filter(
    F.col("status") == "completed"
)

completed_count = completed_orders.count()

print("Completed orders:", completed_count)

assert completed_count == 210_377

Completed orders: 210377


In [18]:
monthly_summary = (
    completed_orders
    .groupBy("month")
    .agg(
        F.sum("revenue_cents").alias("revenue_cents"),
        F.count("*").alias("order_count")
    )
    .orderBy("month")
)

monthly_summary.show(12, truncate=False)

+-------+-------------+-----------+
|month  |revenue_cents|order_count|
+-------+-------------+-----------+
|2025-01|1367416562   |17948      |
|2025-02|1231899944   |16141      |
|2025-03|1347070513   |17901      |
|2025-04|1309972409   |17234      |
|2025-05|1335494910   |17677      |
|2025-06|1312410254   |17160      |
|2025-07|1359078697   |17806      |
|2025-08|1354362185   |18059      |
|2025-09|1302107111   |17183      |
|2025-10|1365367905   |17951      |
|2025-11|1307283134   |17293      |
|2025-12|1363291425   |18024      |
+-------+-------------+-----------+



In [19]:
monthly_summary = (
    completed_orders
    .groupBy("month")
    .agg(
        F.sum("revenue_cents").alias("revenue_cents"),
        F.count("*").alias("order_count")
    )
    .orderBy("month")
)

monthly_summary.show(12, truncate=False)

+-------+-------------+-----------+
|month  |revenue_cents|order_count|
+-------+-------------+-----------+
|2025-01|1367416562   |17948      |
|2025-02|1231899944   |16141      |
|2025-03|1347070513   |17901      |
|2025-04|1309972409   |17234      |
|2025-05|1335494910   |17677      |
|2025-06|1312410254   |17160      |
|2025-07|1359078697   |17806      |
|2025-08|1354362185   |18059      |
|2025-09|1302107111   |17183      |
|2025-10|1365367905   |17951      |
|2025-11|1307283134   |17293      |
|2025-12|1363291425   |18024      |
+-------+-------------+-----------+



In [20]:
product_summary = (
    completed_orders
    .groupBy("product_id", "product_name")
    .agg(
        F.sum("revenue_cents").alias("revenue_cents"),
        F.count("*").alias("order_count")
    )
    .orderBy(F.desc("revenue_cents"), "product_id")
)

product_summary.show(10, truncate=False)

+----------+------------+-------------+-----------+
|product_id|product_name|revenue_cents|order_count|
+----------+------------+-------------+-----------+
|P0048     |Product 48  |38236677     |464        |
|P0266     |Product 266 |36738236     |451        |
|P0220     |Product 220 |36660381     |468        |
|P0436     |Product 436 |36544110     |450        |
|P0197     |Product 197 |36519301     |454        |
|P0426     |Product 426 |36354998     |452        |
|P0093     |Product 93  |36077860     |464        |
|P0183     |Product 183 |35917406     |441        |
|P0453     |Product 453 |35766731     |473        |
|P0486     |Product 486 |35752307     |464        |
+----------+------------+-------------+-----------+
only showing top 10 rows


In [22]:
region_summary = (
    completed_orders
    .groupBy("region")
    .agg(
        F.sum("revenue_cents").alias("revenue_cents"),
        F.count("*").alias("order_count")
    )
    .orderBy(F.desc("revenue_cents"), "region")
)

region_summary.show(truncate=False)

+------+-------------+-----------+
|region|revenue_cents|order_count|
+------+-------------+-----------+
|West  |4006169153   |52797      |
|East  |3999576306   |52759      |
|South |3992654118   |52435      |
|North |3957355472   |52386      |
+------+-------------+-----------+



In [23]:
total_revenue_cents = completed_orders.agg(
    F.sum("revenue_cents")
).first()[0]

for report in [
    monthly_summary,
    region_summary,
    product_summary
]:
    totals = report.agg(
        F.sum("revenue_cents").alias("revenue"),
        F.sum("order_count").alias("orders")
    ).first()

    assert totals["revenue"] == total_revenue_cents
    assert totals["orders"] == completed_count

assert total_revenue_cents == 15_955_755_049

from decimal import Decimal

total_revenue_cad = (
    Decimal(total_revenue_cents) / Decimal(100)
)

print(f"Completed-sales revenue: CAD {total_revenue_cad:,.2f}")
print("All report totals match.")

Completed-sales revenue: CAD 159,557,550.49
All report totals match.


In [24]:
OUTPUT = ROOT / "outputs"

enriched_orders.write.mode("overwrite").partitionBy("month").parquet(
    str(OUTPUT / "clean_orders")
)

rejected_orders.write.mode("overwrite").parquet(
    str(OUTPUT / "rejected_orders")
)

for name, report in [
    ("monthly_summary", monthly_summary),
    ("region_summary", region_summary),
    ("product_summary", product_summary)
]:
    report.write.mode("overwrite").parquet(
        str(OUTPUT / name)
    )

print("Outputs saved.")

Outputs saved.


In [25]:
saved_orders = spark.read.parquet(
    str(OUTPUT / "clean_orders")
)

saved_rejected = spark.read.parquet(
    str(OUTPUT / "rejected_orders")
)

assert saved_orders.count() == clean_count
assert saved_rejected.count() == rejected_count

saved_revenue = (
    saved_orders
    .filter(F.col("status") == "completed")
    .agg(F.sum("revenue_cents"))
    .first()[0]
)

assert saved_revenue == total_revenue_cents

for name in [
    "monthly_summary",
    "region_summary",
    "product_summary"
]:
    saved_report = spark.read.parquet(str(OUTPUT / name))

    totals = saved_report.agg(
        F.sum("revenue_cents").alias("revenue"),
        F.sum("order_count").alias("orders")
    ).first()

    assert totals["revenue"] == total_revenue_cents
    assert totals["orders"] == completed_count

print("Saved data and all report totals verified.")

Saved data and all report totals verified.


In [26]:
import shutil
from google.colab import files

results_zip = shutil.make_archive(
    "/content/sales_results",
    "zip",
    str(OUTPUT)
)

files.download(results_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
monthly_summary.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (34)
+- Sort (33)
   +- Exchange (32)
      +- HashAggregate (31)
         +- Exchange (30)
            +- HashAggregate (29)
               +- Project (28)
                  +- BroadcastHashJoin Inner BuildRight (27)
                     :- Project (23)
                     :  +- BroadcastHashJoin Inner BuildRight (22)
                     :     :- Project (18)
                     :     :  +- Filter (17)
                     :     :     +- BroadcastHashJoin LeftOuter BuildRight (16)
                     :     :        :- Project (11)
                     :     :        :  +- BroadcastHashJoin LeftOuter BuildRight (10)
                     :     :        :     :- HashAggregate (5)
                     :     :        :     :  +- Exchange (4)
                     :     :        :     :     +- HashAggregate (3)
                     :     :        :     :        +- Filter (2)
                     :     :        :     :           +- Scan csv  (1)
     

In [28]:
monthly_summary.show(12, truncate=False)

+-------+-------------+-----------+
|month  |revenue_cents|order_count|
+-------+-------------+-----------+
|2025-01|1367416562   |17948      |
|2025-02|1231899944   |16141      |
|2025-03|1347070513   |17901      |
|2025-04|1309972409   |17234      |
|2025-05|1335494910   |17677      |
|2025-06|1312410254   |17160      |
|2025-07|1359078697   |17806      |
|2025-08|1354362185   |18059      |
|2025-09|1302107111   |17183      |
|2025-10|1365367905   |17951      |
|2025-11|1307283134   |17293      |
|2025-12|1363291425   |18024      |
+-------+-------------+-----------+



In [29]:
from google.colab import output
from urllib.parse import urlparse

ui_url = spark.sparkContext.uiWebUrl

if ui_url:
    output.serve_kernel_port_as_iframe(
        urlparse(ui_url).port,
        height=600
    )
else:
    print("Spark UI is unavailable in this session.")

<IPython.core.display.Javascript object>

In [30]:
from time import perf_counter
from statistics import median

def calculate_monthly(df):
    return (
        df.groupBy("month")
        .agg(
            F.sum("revenue_cents").alias("revenue_cents"),
            F.count("*").alias("order_count")
        )
        .orderBy("month")
    )

def timed_run(df):
    start = perf_counter()
    result = calculate_monthly(df).collect()
    return perf_counter() - start, result

In [31]:
spark.catalog.clearCache()

benchmark_df = completed_orders.select("month", "revenue_cents")

timed_run(benchmark_df)  # Warm-up run, excluded from timings

uncached_times = []

for _ in range(3):
    seconds, baseline_result = timed_run(benchmark_df)
    uncached_times.append(seconds)

print("Uncached times:", uncached_times)

Uncached times: [2.775622757000292, 3.991437068000323, 3.480519287000334]


In [32]:
cached_df = benchmark_df.cache()

start = perf_counter()
cached_df.count()
cache_setup_seconds = perf_counter() - start

timed_run(cached_df)  # Warm-up run

cached_times = []

for _ in range(3):
    seconds, cached_result = timed_run(cached_df)
    cached_times.append(seconds)

    assert cached_result == baseline_result

cached_df.unpersist()

DataFrame[month: string, revenue_cents: bigint]

In [33]:
uncached_median = median(uncached_times)
cached_median = median(cached_times)

print(f"Cache setup: {cache_setup_seconds:.3f} seconds")
print(f"Uncached median: {uncached_median:.3f} seconds")
print(f"Cached median: {cached_median:.3f} seconds")
print("Results match.")

if cached_median < uncached_median:
    print("Cached runs were faster in this experiment.")
else:
    print("Caching did not improve the median time in this experiment.")

Cache setup: 5.302 seconds
Uncached median: 3.481 seconds
Cached median: 1.015 seconds
Results match.
Cached runs were faster in this experiment.


In [34]:
import json

performance_results = {
    "environment": "Google Colab",
    "spark_version": spark.version,
    "spark_master": spark.sparkContext.master,
    "raw_order_rows": raw_count,
    "completed_order_rows": completed_count,
    "shuffle_partitions": spark.conf.get(
        "spark.sql.shuffle.partitions"
    ),
    "calculation": "Monthly completed-sales aggregation",
    "measured_runs_per_condition": 3,
    "warmup_runs_excluded": True,
    "uncached_times_seconds": uncached_times,
    "cached_times_seconds": cached_times,
    "uncached_median_seconds": uncached_median,
    "cached_median_seconds": cached_median,
    "cache_setup_seconds": cache_setup_seconds,
    "results_match": cached_result == baseline_result,
    "limitation": (
        "Single Colab local-mode experiment. "
        "Timings may vary; cache setup is measured separately."
    )
}

OUTPUT.mkdir(parents=True, exist_ok=True)

performance_file = OUTPUT / "performance_results.json"

performance_file.write_text(
    json.dumps(performance_results, indent=2)
)

print(performance_file.read_text())

{
  "environment": "Google Colab",
  "spark_version": "4.0.1",
  "spark_master": "local[2]",
  "raw_order_rows": 250000,
  "completed_order_rows": 210377,
  "shuffle_partitions": "8",
  "calculation": "Monthly completed-sales aggregation",
  "measured_runs_per_condition": 3,
  "warmup_runs_excluded": true,
  "uncached_times_seconds": [
    2.775622757000292,
    3.991437068000323,
    3.480519287000334
  ],
  "cached_times_seconds": [
    1.1633438250000836,
    0.9801905979998082,
    1.0153301049999754
  ],
  "uncached_median_seconds": 3.480519287000334,
  "cached_median_seconds": 1.0153301049999754,
  "cache_setup_seconds": 5.301934863000042,
  "results_match": true,
  "limitation": "Single Colab local-mode experiment. Timings may vary; cache setup is measured separately."
}


In [35]:
results_zip = shutil.make_archive(
    "/content/sales_results",
    "zip",
    str(OUTPUT)
)

files.download(results_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
readme_text = """# PySpark Sales Data Engineering Pipeline

Processes 250,000 simulated sales records into validated
Parquet datasets and sales reports.

## Business problem

A fictional retailer needs reliable monthly, regional,
and product sales reports from messy order files.

## Pipeline

CSV inputs → Deduplication → Validation → Customer/product joins
→ Sales summaries → Parquet outputs → Verification

## Data

- 250,000 raw order rows across five CSV files
- 10,000 customers
- 500 products
- All data is simulated
- Monetary values are stored as integer Canadian cents

## Cleaning rules

Remove exact duplicates. Reject orders with invalid customer
or product IDs, non-positive quantities, negative prices,
invalid dates, unsupported statuses, or non-CAD currency.

Rejected records are retained with a reason for review.

## Expected results

The project reference checks specify:
- 1,000 exact duplicates removed
- 1,494 invalid orders rejected
- 247,506 valid orders
- 210,377 completed orders
- CAD 159,557,550.49 in completed-sales revenue

The notebook contains assertions to compare actual results
against these values and verify that report totals agree.

Cancelled and refunded orders are excluded from sales reports.
This is completed-sales revenue, not net revenue or profit.

## Outputs

- Clean enriched orders, partitioned by month
- Rejected orders with rejection reasons
- Monthly sales summary
- Regional sales summary
- Product sales summary
- Performance measurements in JSON

Datasets and summaries are saved as Parquet.

## Performance experiment

The same monthly aggregation was measured in Colab local mode.
One warm-up run per condition was excluded, followed by three
measured runs per condition.

- Uncached median: 3.481 seconds
- Cached median: 1.015 seconds
- Cache population: 5.302 seconds
- Aggregation results matched

Cached execution had approximately 70.8% lower median runtime
in this experiment. Cache setup adds an initial cost.
These timings do not demonstrate multi-machine cluster scaling.

## How to run

1. Open the executed project notebook in Google Colab.
2. Run the PySpark installation cell.
3. Upload Sales_Data_Engineering_250000.zip when prompted.
4. Run the remaining cells in order.
5. Download the output archive and executed notebook.

## Verification

The notebook checks:
- Unique customer and product keys
- Unique order IDs after deduplication
- Row conservation during validation
- Unchanged row counts after joins
- Consistent revenue and order counts across reports
- Counts and totals after reading saved Parquet files
- Matching cached and uncached aggregation results

## Credits and limitations

The simulated dataset, starter scaffold, and guided code were
provided with ChatGPT assistance. I ran the pipeline in Colab.

This is an educational batch pipeline. It does not yet include
production scheduling, monitoring alerts, or cloud deployment.
"""

readme_file = ROOT / "README.md"
readme_file.write_text(readme_text)

print("README created.")

README created.


In [37]:
files.download(str(readme_file))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>